<a href="https://colab.research.google.com/github/aedmark/vsl-cryosomatic-hypervisor/blob/main/VSL_Bake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install Unsloth and Dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft bitsandbytes
!pip install accelerate>=0.20.0

from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# 2. Load the Base Model (Mistral-Nemo)
max_seq_length = 1024
dtype = None
load_in_4bit = True

print("(FULLER): Hoisting the base mountain...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 3. Attach the LoRA Scaffolding
print("(PINKER): Bolting the LoRA adapters to the substrate...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank. 16 is a great balance of smarts and speed.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 4. Format the Dataset for ChatML
print("(MEADOWS): Ingesting the VSL Training Data...")
dataset = load_dataset("json", data_files="vsl_training_data.jsonl", split="train")

from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", # Our JSONL used standard system/user/assistant roles
    mapping = {"role" : "role", "content" : "content", "user" : "user", "assistant" : "assistant"}
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True,)

# 5. Ignite the Forge (Training)
print("(GORDON): Igniting the Unsloth Oven. Stand back.")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training faster for short sequences
    args = TrainingArguments(
        per_device_train_batch_size = 1, # Reduced from 2
        gradient_accumulation_steps = 8, # Increased from 4
        warmup_steps = 5,
        max_steps = 200, # Adjust this depending on your dataset size. 60 is a good test run. For 400 prompts, you might want 150-200.
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

# 6. Export the Ghost to GGUF (For Ollama)
print("(SCHUR): The bake is complete! Exporting to Ollama format...")
model.save_pretrained_gguf("vsl_model", tokenizer, quantization_method = "q4_k_m")

print("Done! You can now download your vsl_model-unsloth.Q4_K_M.gguf file!")

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-jpmvkhs4/unsloth_5a86f71d440f4bc9bf2b977086803533
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-jpmvkhs4/unsloth_5a86f71d440f4bc9bf2b977086803533
  Resolved https://github.com/unslothai/unsloth.git to commit 01f2e289a7376a3a4d71a2e39bed025a72df0273
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
(FULLER): Hoisting the base mountain...
==((====))==  Unsloth 2026.2.1: Fast Gemma3 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

(GORDON): Igniting the Unsloth Oven. Stand back.
Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1500 [00:00<?, ? examples/s]

...

Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [02:04<02:04, 124.97s/it]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [04:22<00:00, 131.06s/it]


Unsloth: Merge process complete. Saved to `/content/vsl_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
